# "Trung bình mỗi ngày bán bao nhiêu?" Lưới thời gian, resample tuần và kỳ chưa đủ trên Online Retail II

Bộ `online-retail-ii`, sheet **`Year 2010-2011`** (541 910 dòng, 01/12/2010 tới 09/12/2011): cửa hàng quà tặng
online ở Anh. Tháng 12/2011 chỉ có dữ liệu tới ngày 9.

**Bối cảnh.** Trưởng bộ phận vận hành lập kế hoạch nhân sự kho cho năm sau và hỏi hai câu: *"Trung bình mỗi ngày
mình bán bao nhiêu?"* và *"Gửi tôi doanh thu theo tuần kèm đường xu hướng 4 tuần"*. Dữ liệu giao dịch chỉ có dòng
cho ngày có đơn; ngày không có dòng có thể là ngày nghỉ, ngày lễ, hay ngày sau cùng của dữ liệu. Bạn phải đặt dữ liệu
lên **lưới lịch** đầy đủ trước khi tính.

Quy tắc làm sạch dùng chung của track: đổi `Invoice` sang `str`, bỏ hoá đơn bắt đầu bằng `C`, bỏ dòng
`Quantity <= 0` hoặc `Price <= 0`. Doanh thu dòng = `Quantity * Price` (GBP, doanh thu gộp).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/04-python-for-analysts/exercises/basic/03-retail-daily-weekly -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc sheet 2010-2011 (xlsx: khoảng 30-60 giây)
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

RETAIL = ensure("online-retail-ii")
tx = pd.read_excel(RETAIL / "online_retail_II.xlsx", sheet_name="Year 2010-2011")
print(tx.shape, tx.InvoiceDate.min(), "→", tx.InvoiceDate.max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Doanh thu theo ngày trên lưới lịch

- `clean_sales(tx) -> pd.DataFrame`: các dòng hợp lệ (theo quy tắc trên, `Invoice` là `str`) kèm cột `revenue`.
- `daily_revenue(tx) -> pd.Series`: doanh thu mỗi **ngày lịch** từ ngày của dòng hợp lệ đầu tiên tới ngày của dòng
  hợp lệ cuối cùng, kể cả ngày không có đơn (giá trị `0.0`). Index là `DatetimeIndex` tần suất ngày (`freq="D"`),
  tên `date`, mỗi phần tử là 00:00 của ngày đó; Series tên `revenue`.

In [ ]:
def clean_sales(tx: pd.DataFrame) -> pd.DataFrame:
    """Dòng hợp lệ (Invoice là str) kèm cột revenue = Quantity * Price; không sửa tx."""
    # TODO
    raise NotImplementedError


def daily_revenue(tx: pd.DataFrame) -> pd.Series:
    """Doanh thu mỗi ngày lịch, ngày không có đơn = 0; index date (freq D), tên revenue."""
    # TODO: set_index("InvoiceDate") rồi resample("D")
    raise NotImplementedError

In [ ]:
# explore: những ngày không có đơn rơi vào thứ mấy?
daily = daily_revenue(tx)
print(len(daily), "ngày lịch,", int((daily > 0).sum()), "ngày có đơn")
zero = daily[daily == 0]
print(zero.index.day_name().value_counts())
print("ngày không bán mà không phải thứ Bảy:")
print(zero[zero.index.day_of_week != 5].index.strftime("%Y-%m-%d %a").tolist())

## 2. Trung bình mỗi ngày: mẫu số nào?

`daily_average(daily) -> pd.Series`: bốn số, index theo thứ tự `calendar_days` (số ngày lịch), `trading_days` (số
ngày doanh thu `> 0`), `per_calendar_day` (tổng doanh thu / ngày lịch), `per_trading_day` (tổng doanh thu / ngày có
bán). Series tên `daily_average`.

In [ ]:
def daily_average(daily: pd.Series) -> pd.Series:
    """calendar_days, trading_days, per_calendar_day, per_trading_day; tên daily_average."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: hai mẫu số, và trung bình theo thứ trong tuần
avg = daily_average(daily)
print(avg.round(1))
print(f"chênh: {avg.per_trading_day / avg.per_calendar_day - 1:.1%}")
by_dow = daily.groupby(daily.index.day_of_week).agg(["mean", "size"])
by_dow.index = ["T2", "T3", "T4", "T5", "T6", "T7", "CN"]
print(by_dow.round(0))

## 3. Báo cáo tuần có cờ "chưa đủ"

`weekly_report(daily) -> pd.DataFrame`: gộp `daily` theo tuần kết thúc Chủ nhật (`resample("W-SUN")`, index là ngày
Chủ nhật, tên `week_end`). Năm cột theo thứ tự:

| Cột | Định nghĩa |
| --- | --- |
| `revenue` | tổng doanh thu tuần |
| `trading_days` | số ngày của tuần có doanh thu `> 0` |
| `calendar_days` | số ngày của tuần nằm trong `daily` (tuần đầu, tuần cuối có thể < 7) |
| `complete` | `True` khi `calendar_days == 7` |
| `rolling_4w` | trung bình `revenue` của 4 tuần **đủ** liền nhau kết thúc ở tuần đó; tuần chưa đủ thì `NaN`, và 4 tuần nào chứa một tuần chưa đủ cũng ra `NaN` |

Gợi ý cho `rolling_4w`: thay doanh thu tuần chưa đủ bằng `NaN` (`revenue.where(complete)`) rồi `rolling(4).mean()`.

In [ ]:
def weekly_report(daily: pd.Series) -> pd.DataFrame:
    """Index week_end (Chủ nhật); revenue, trading_days, calendar_days, complete, rolling_4w."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: bảng tuần, tuần đầu/cuối, các tuần cao nhất
weekly = weekly_report(daily)
print(weekly.head(6).round(0).to_string())
print(weekly.tail(3).round(0).to_string())
print(weekly.nlargest(3, "revenue").round(0).to_string())

In [ ]:
# explore: tuần cuối cao bất thường - hoá đơn lớn nhất tuần đó là gì, và chuyện gì xảy ra ngay sau?
sales = clean_sales(tx)
last = sales[sales.InvoiceDate >= "2011-12-05"].groupby("Invoice").agg(
    revenue=("revenue", "sum"), customer=("Customer ID", "first"), time=("InvoiceDate", "first"))
print(last.nlargest(3, "revenue"))
print(tx[tx["Invoice"].astype(str).isin(["581483", "C581484"])][["Invoice", "Description", "Quantity", "Price", "InvoiceDate"]])

In [ ]:
# explore: doanh thu tuần và đường 4 tuần; tuần chưa đủ vẽ màu xám
fig, ax = plt.subplots(figsize=(10, 4))
colors = np.where(weekly.complete, "#4c78a8", "#c0c7d0")
ax.bar(weekly.index, weekly.revenue / 1e3, width=5, color=colors)
ax.plot(weekly.index, weekly.rolling_4w / 1e3, color="#e45756", lw=2, label="trung bình 4 tuần đủ")
ax.set_ylabel("nghìn GBP / tuần")
ax.legend(frameon=False, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Doanh thu tuần: tuần cao nhất là tuần chưa đủ (xám) cuối dữ liệu", loc="left")
plt.show()

## 4. Đọc kết quả, rồi làm trắc nghiệm

Trả lời trưởng bộ phận vận hành bằng ba câu:

1. "Trung bình mỗi ngày" là bao nhiêu, theo mẫu số nào, và vì sao bạn chọn mẫu số đó cho kế hoạch nhân sự kho?
2. Những ngày không có đơn gồm những loại nào (thứ trong tuần, lễ)? Ngày nào trong tuần ít đơn nhất trong số ngày
   có mở?
3. Tuần nào doanh thu cao nhất? Con số đó có nên đưa vào báo cáo xu hướng không?

Xong thì mở app, làm phần **trắc nghiệm** của bài.